In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import copy

from tqdm import tqdm
from docx import Document
from docx.shared import Inches
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT, WD_ALIGN_PARAGRAPH

import Flexivan_Prediction_Package

## Daily Accum Report Accuracy Check
##### This notebook contains the development of the accuracy check. The product is a function in the main package
##### (Flexivan_Prediction_Package). It is insignificant for final product version

In [2]:
# Get filenames from folder
print('Getting PAST raw data filenames...')
# DATA_Folder = '/root/Flexivan/Flexivan/Daily prediction/DATA'
DATA_Folder = '/root/Flexivan/Flexivan/inference_data_Oct_to_Dec_2025'

FILENAMES = [f for f in os.listdir(DATA_Folder) if os.path.isfile(os.path.join(DATA_Folder, f))]
selected = "Latest_Test_"
selected2 = '_DETAILED'
FILENAMES = [f for f in FILENAMES if selected in f and selected2 not in f]
DATES = Flexivan_Prediction_Package.extract_datetimes_from_filenames(FILENAMES)
Filenames_DF = pd.DataFrame({
    "Filenames": FILENAMES,
    "Dates": DATES
})
Filenames_DF_Sorted = Filenames_DF.sort_values(by='Dates')
FILENAMES = list(Filenames_DF_Sorted['Filenames'])
DATES = list(Filenames_DF_Sorted['Dates'])

print(f'DONE - {len(FILENAMES)} Filenames were found')

#endregion

Getting PAST raw data filenames...
DONE - 71 Filenames were found


In [3]:
def Get_Filename_by_Date(FILENAMES, Date_STR):

    for filename in FILENAMES:
        ARGS = filename.split('_')
        if ARGS[2] == Date_STR:
            return filename

    return None             # Date not found in FILENAMES

def Check_Daily_ACCUM_Report_Accuracy(Daily_ACCUM_Report_DF, LOT_Name, FILENAMES, DATA_Folder):
    # This function goes through the dates of the Daily_ACCUM_Report_DF and checks the number of returns and pickups
    # in actual file of the same date

    DATES = list(Daily_ACCUM_Report_DF['Date'].unique())
    
    for DATE in DATES:
        PUs_Pred = Daily_ACCUM_Report_DF.index(DATE)['Predicted_Pickups']
        Returns_Pred = Daily_ACCUM_Report_DF.index(DATE)['Predicted_Returns']

        # Get actual pickups and returns from file

    return Accuracy_DF

def Calculate_PUs_Returns_by_Dates(FOLDER, FILENAMES):
    # This function goes through all the selected FILENAMES in FOLDER and retrieves the final number of PUs and RETs
    Pickup_Date_Field = 'CHS Pickup Date'
    Return_Date_Field = 'CHS Return Dt' 

    DATES_PUs_RETs_Sums_DF = pd.DataFrame(columns=['Total_PUs', 'Total_Returns'])
    DATES_PUs_RETs_Sums_DF.index.name = 'Date'

    print('Collecting PUs and Returns sums from all files...')

    for filename in tqdm(FILENAMES):
        DATA = pd.read_csv(f'{FOLDER}/{filename}')

        try:
            DATA[Pickup_Date_Field] = pd.to_datetime(DATA[Pickup_Date_Field])
        except:
            pass

        try:
            DATA[Return_Date_Field] = pd.to_datetime(DATA[Return_Date_Field])
        except:
            pass

        DATA[Pickup_Date_Field] = DATA[Pickup_Date_Field].dt.date
        DATA[Return_Date_Field] = DATA[Return_Date_Field].dt.date

        Filename_Dates = list(DATA[Pickup_Date_Field].unique()) + list(DATA[Return_Date_Field].unique())
        Filename_Dates = list(set(Filename_Dates))

        for DATE in Filename_Dates:
            #region Check if date exists
            if DATE not in DATES_PUs_RETs_Sums_DF.index:
                DATES_PUs_RETs_Sums_DF.loc[DATE] = {'Total_PUs': 0, 'Total_Returns': 0}

            #endregion

            DATA_TEMP = DATA[DATA[Pickup_Date_Field]==DATE]
            DATES_PUs_RETs_Sums_DF.loc[DATE, 'Total_PUs'] += len(DATA_TEMP)

            DATA_TEMP = DATA[DATA[Return_Date_Field]==DATE]
            DATES_PUs_RETs_Sums_DF.loc[DATE, 'Total_Returns'] += len(DATA_TEMP)

    print('DONE.')

    DATES_PUs_RETs_Sums_DF = DATES_PUs_RETs_Sums_DF.sort_values(by='Date')
        
    return DATES_PUs_RETs_Sums_DF

In [4]:
# Collect all PUs and Returns from all data files
DATES_PUs_RETs_Sums_DF = Calculate_PUs_Returns_by_Dates(DATA_Folder, FILENAMES)
DATES_PUs_RETs_Sums_DF

100%|██████████| 71/71 [00:37<00:00,  1.90it/s]

DONE.


,Total_PUs,Total_Returns
Date,,
2023-04-12,1,0
2024-04-22,14,0
2024-04-23,4,0
2024-05-01,8,0
2024-05-07,17,0
...,...,...
2025-12-06,33,142
2025-12-08,158,201
2025-12-09,36,407


In [5]:
DATES_PUs_RETs_Sums_DF.to_csv('KILLERS.csv')


In [ ]:
# Load daily report(s) for accuracy check
Reports_Folder = '/root/Flexivan/Flexivan/DAILY_ACCUM_REPORTS'

# Get all reports from folder
REPORT_FILENAMES = [f for f in os.listdir(Reports_Folder) if os.path.isfile(os.path.join(Reports_Folder, f))]

Reports_Accuracy_DF = pd.DataFrame(columns=['Report_Filename', 'PUs_Pred','PUs_GT', 'Returns_Pred', 'Returns_GT'])
print('Checking accuracy for reports in selected folder...')

DATES_PUs_RETs_Sums_DF.index = pd.to_datetime(DATES_PUs_RETs_Sums_DF.index).strftime('%Y-%m-%d')

for report_filename in tqdm(REPORT_FILENAMES):
    REPORT = pd.read_csv(f'{Reports_Folder}/{report_filename}', index_col=False)
    REPORT['Date'] = REPORT[REPORT.columns[0]]
    Report_Dates = list(REPORT['Date'].unique())
    REPORT = REPORT.set_index('Date')

    for report_date in Report_Dates:
        try:
            PUs_Number_Pred = REPORT.loc[report_date]['Predicted_Pickups']
            Rets_Number_Pred = REPORT.loc[report_date]['Predicted_Returns']

            PUs_Number_GT = DATES_PUs_RETs_Sums_DF.loc[report_date]['Total_PUs']
            Rets_Number_GT = DATES_PUs_RETs_Sums_DF.loc[report_date]['Total_Returns']

            ROW = [report_filename, PUs_Number_Pred, PUs_Number_GT, Rets_Number_Pred, Rets_Number_GT]
            Reports_Accuracy_DF.loc[len(Reports_Accuracy_DF)] = ROW
        except:
            pass

print('DONE.')

Checking accuracy for reports in selected folder...


  0%|          | 0/17 [00:00<?, ?it/s]

In [24]:
Reports_Accuracy_DF

,Report_Filename,PUs_Pred,PUs_GT,Returns_Pred,Returns_GT
0,Daily_ACCUM_Report_LAXCSN_2025-12-09.csv,119,8588,99,3216
1,Daily_ACCUM_Report_LAXCSN_2025-12-09.csv,499,9057,478,6495
2,Daily_ACCUM_Report_LAXCSN_2025-12-09.csv,692,2189,681,3272
3,Daily_ACCUM_Report_LAXCSN_2025-12-09.csv,835,5919,823,7296
4,Daily_ACCUM_Report_LAXCSN_2025-12-09.csv,1026,4859,1015,12307
...,...,...,...,...,...
611,Daily_ACCUM_Report_LAXEMS_2025-12-09.csv,35,5951,35,4742
612,Daily_ACCUM_Report_LAXEMS_2025-12-09.csv,35,4959,35,2966
613,Daily_ACCUM_Report_LAXEMS_2025-12-09.csv,35,902,35,1203
614,Daily_ACCUM_Report_LAXEMS_2025-12-09.csv,35,2704,35,2886
